# KMeans Customer Segmentation

Simple unsupervised customer segmentation using customer-level KPIs.

The notebook searches between 5 and 7 clusters, then profiles each cluster graphically. It uses synthetic data by default, so it stays generic and safe for a public repo.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from analytics_decision_kit.sample_data import create_demo_orders
from analytics_decision_kit.customer_analysis import calculate_customer_metrics
from analytics_decision_kit.kmeans_segmentation import run_kmeans_segmentation


## 1. Create customer metrics


In [ ]:
orders = create_demo_orders(n_customers=5000, n_orders=18000, seed=42)
customers = calculate_customer_metrics(orders)

customers.head()


## 2. Run KMeans with 5 to 7 clusters


In [ ]:
result = run_kmeans_segmentation(customers, k_min=5, k_max=7)

print('Selected k:', result.selected_k)
result.diagnostics


## 3. Cluster profile


In [ ]:
profile = result.cluster_profile.copy()
profile


## 4. Customer share vs revenue share


In [ ]:
plot_df = profile.sort_values('revenue_share', ascending=True)

ax = plot_df.set_index('cluster_name')[['customer_share', 'revenue_share']].plot(
    kind='barh',
    figsize=(10, 5),
)
ax.set_title('Cluster size vs revenue contribution')
ax.set_xlabel('Share')
ax.set_ylabel('Cluster')
ax.legend(['Customer share', 'Revenue share'])
plt.tight_layout()
plt.show()


## 5. Average revenue and orders


In [ ]:
plot_df = profile.sort_values('avg_revenue', ascending=True)

ax = plot_df.plot(
    x='cluster_name',
    y='avg_revenue',
    kind='barh',
    figsize=(10, 5),
    legend=False,
)
ax.set_title('Average revenue per customer by cluster')
ax.set_xlabel('Average revenue')
ax.set_ylabel('Cluster')
plt.tight_layout()
plt.show()


In [ ]:
plot_df = profile.sort_values('avg_orders', ascending=True)

ax = plot_df.plot(
    x='cluster_name',
    y='avg_orders',
    kind='barh',
    figsize=(10, 5),
    legend=False,
)
ax.set_title('Average orders by cluster')
ax.set_xlabel('Average orders')
ax.set_ylabel('Cluster')
plt.tight_layout()
plt.show()


## 6. Feature profile heatmap


In [ ]:
features = result.cluster_features.merge(
    profile[['cluster', 'cluster_name']],
    on='cluster',
    how='left',
).set_index('cluster_name')

features = features[result.features]

# Min-max scale for quick visual comparison inside the notebook
scaled_features = (features - features.min()) / (features.max() - features.min())
scaled_features = scaled_features.fillna(0)

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(scaled_features.values, aspect='auto')
ax.set_xticks(range(len(scaled_features.columns)))
ax.set_xticklabels(scaled_features.columns, rotation=45, ha='right')
ax.set_yticks(range(len(scaled_features.index)))
ax.set_yticklabels(scaled_features.index)
ax.set_title('Relative feature profile by cluster')
fig.colorbar(im, ax=ax, label='Relative intensity')
plt.tight_layout()
plt.show()


## 7. Sample customers with assigned clusters


In [ ]:
result.customers[[
    'customer_id',
    'revenue',
    'orders',
    'avg_order_value',
    'cluster',
    'cluster_name',
]].head(20)
